In [16]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video
from imaginaire.lazy_config import instantiate

from cosmos_predict2.configs.expert.defaults.data_calvin import calvin_train_dataset, DataLoader, runnable_train_transforms

In [17]:
from cosmos_predict2.configs.expert.defaults.data_calvin import transforms_modified_by_stat
print(type(instantiate(transforms_modified_by_stat)))

<class 'omegaconf.dictconfig.DictConfig'>


In [18]:
train_transforms = instantiate(runnable_train_transforms)
print(train_transforms)

{'rgb_static': Compose(
    Resize(size=[224, 224], interpolation=bilinear, max_size=None, antialias=True)
    RandomShiftsAug()
), 'gen_static': Compose(
    Resize(size=[224, 224], interpolation=bilinear, max_size=None, antialias=True)
), 'rgb_gripper': Compose(
    Resize(size=[224, 224], interpolation=bilinear, max_size=None, antialias=True)
    RandomShiftsAug()
), 'gen_gripper': Compose(
    Resize(size=[224, 224], interpolation=bilinear, max_size=None, antialias=True)
), 'depth_static': Compose(
    Resize(size=200, interpolation=bilinear, max_size=None, antialias=True)
    AddDepthNoise(shape=tensor([1000.]), rate=tensor([1000.]), dist=Gamma(concentration: tensor([1000.]), rate: tensor([1000.])))
    AddGaussianNoise(mean=tensor([0.]), std=tensor([0.0100]))
), 'depth_gripper': Compose(
    Resize(size=84, interpolation=bilinear, max_size=None, antialias=True)
    AddGaussianNoise(mean=tensor([0.]), std=tensor([0.0100]))
), 'rgb_tactile': Compose(
    Resize(size=70, interpolati

In [19]:
train_dataset = instantiate(
    calvin_train_dataset,
    future_frame_skip=3,
)
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=16,
    drop_last=True
)
# print("====" * 10)
# print(train_dataset.transforms)

[DEBUG] loading dataset at /home/geyuan/code/mdt24rss_fork/dataset/task_D_D/training
trying to load lang data from:  /home/geyuan/code/mdt24rss_fork/dataset/task_D_D/training/lang_clip_resnet50/auto_lang_ann.npy
[DEBUG] LANG ep_start_end_ids: 5124 (315660, 315724) (191730, 191794) (305439, 305503)
[DEBUG] LANG episode_lookup len= 314042
[DEBUG] Extracted files loaded from /home/geyuan/code/mdt24rss_fork/dataset/task_D_D/training/extracted
[DEBUG] Loading language embeddings from: /home/geyuan/code/mdt24rss_fork/dataset/task_D_D/lang_emb_t5xxl/t5_embeddings.npz


In [20]:
from tqdm import tqdm

vis_idx = 3
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch('CALVIN@0', batch)
    if idx < vis_idx:
        continue
    print_batch(f'CALVIN@{idx}', batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                                                     | 1/39255 [00:00<9:30:31,  1.15it/s]

CALVIN@0: Dict, keys=['action', 'video', 'agent_pos', 'annotation_file', '__key__', 'lang_text', 't5_text_embeddings', 't5_text_mask', 'fps', 'image_size', 'num_frames', 'padding_mask', 'sample_n_views', 'view_indices', 'latent_view_indices_B_T']
--action, <class 'torch.Tensor'>, shape=torch.Size([8, 12, 7]), min=-0.7764, max=1.0000
--video, <class 'torch.Tensor'>, shape=torch.Size([8, 3, 10, 224, 224]), min=0.0000, max=255.0000
--agent_pos, <class 'torch.Tensor'>, shape=torch.Size([8, 5, 8]), min=-0.8086, max=1.4233
--annotation_file: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--__key__: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=4, value='None'
--lang_text: List, len=8, elem_type=<class 'str'>
----[0]: <class 'str'>, len=30, value='move the door to the left side'
--t5_text_embeddings, <class 'torch.Tensor'>, shape=torch.Size([8, 512, 1024]), min=-0.6016, max=0.5859
--t5_text_mask, <class 'torch.Tensor'>, shape=torch.Size([8,

  0%|                                                                                     | 3/39255 [00:01<5:33:07,  1.96it/s]


In [21]:
print(in_sample["view_indices"][0])
print(in_sample["latent_view_indices_B_T"][0])

tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
tensor([0, 0, 1, 1])


In [22]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample
DEBUG_DATASET = "calvin"

horizon = mv_sample['agent_pos'][0].shape[0]
save_image_or_video(
    mv_sample['video'][0, :, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=5
)
save_image_or_video(
    mv_sample['video'][0, :, horizon:].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_gripper.mp4",
    fps=5
)

# dataset_stats = train_dataset.dataset_stats

save_action_as_image(
    train_dataset.denorm_action(
        mv_sample['action'].cpu().numpy()
    )[0, :, :7],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action0123456.mp4",
)
# save_action_as_image(
#     train_dataset.denorm_action(
#         mv_sample['action'], train_dataset.norm_action_type, dataset_stats['rel_actions']
#     )[0, :, 6:7],
#     f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
# )

save_action_as_image(
    train_dataset.denorm_state_or_force(
        mv_sample['agent_pos'].cpu().numpy()
    )[0, :, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )


Plotting action dynamic figures...
[DEBUG] save_3d_action_as_image: (5, 3) float32 -0.21286364 1.0307546
[11-01 18:15:26|INFO|../../../../../../../home/geyuan/code/cospred2nvidia/cosmos_predict2/utils/vis_helpers.py:197:save_3d_action_as_image] Saved 3D action trajectory visualization to /home/geyuan/code/cospred2nvidia/output/de_calvin_mv_agentpos012.png
